### Pyspark GraphX

### Step 1: Set Up Spark Session
Create a Spark session to initialize the distributed computing environment

In [1]:
from pyspark.sql import SparkSession
# Initialize Spark session
spark = SparkSession.builder \
.appName("GraphXShortestPathLab") \
.config("spark.driver.memory", "4g") \
.config("spark.jars", "/Users/rattanak/jars/graphframes-0.8.4-spark3.5-s_2.12.jar") \
.getOrCreate()
# .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.5-s_2.12") \
sc = spark.sparkContext

25/04/20 15:04:02 WARN Utils: Your hostname, Rattanaks-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.17 instead (on interface en0)
25/04/20 15:04:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/04/20 15:04:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Step 2: Create a Directed Graph
Define a small directed graph representing a social network where nodes are people (e.g., Alice,
Bob) and edges represent directed relationships (e.g., Alice follows Bob) with weights (e.g.,
interaction strength).

In [2]:
from pyspark.sql import Row
# Define vertices (nodes) as a DataFrame
vertices = spark.createDataFrame([
 Row(id=1, name="Alice"),
 Row(id=2, name="Bob"),
 Row(id=3, name="Charlie"),
 Row(id=4, name="David"),
 Row(id=5, name="Eve")
])
# Define edges with weights
edges = spark.createDataFrame([
 Row(src=1, dst=2, weight=1.0), # Alice -> Bob
 Row(src=2, dst=3, weight=2.0), # Bob -> Charlie
 Row(src=1, dst=3, weight=4.0), # Alice -> Charlie
 Row(src=3, dst=4, weight=1.0), # Charlie -> David
 Row(src=2, dst=4, weight=3.0), # Bob -> David
 Row(src=4, dst=5, weight=2.0), # David -> Eve
 Row(src=3, dst=5, weight=5.0) # Charlie -> Eve
])
# Cache DataFrames for performance
vertices.cache()
edges.cache()

DataFrame[src: bigint, dst: bigint, weight: double]

### Step 3: Build GraphX Graph
Convert the DataFrames into a GraphX graph using the graphframes library, which provides a
Python interface for GraphX.

In [3]:
from graphframes import GraphFrame
# Create GraphFrame
graph = GraphFrame(vertices, edges)
# # Display graph structure
print("Vertices:")
graph.vertices.show()
print("Edges:")
graph.edges.show()

/Users/rattanak/miniconda3/envs/py312/lib/python3.12/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


Vertices:


+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
|  4|  David|
|  5|    Eve|
+---+-------+

Edges:
+---+---+------+
|src|dst|weight|
+---+---+------+
|  1|  2|   1.0|
|  2|  3|   2.0|
|  1|  3|   4.0|
|  3|  4|   1.0|
|  2|  4|   3.0|
|  4|  5|   2.0|
|  3|  5|   5.0|
+---+---+------+

